**Qwen 系列模型工具调用与推理能力评估**

本文对 **Qwen2.5** 与 **Qwen3** 系列开源模型进行了系统性的实测评估，重点关注其在 **工具调用（Tool Use）** 与 **长链推理（Long-Horizon Reasoning）** 场景下的表现。

本次评测涵盖以下基准数据集：

* **BFCL-v4**：一个面向工具调用能力的综合评测基准，覆盖单工具、多工具、并行调用、多轮交互、网页检索与记忆等多种复杂场景，并基于 **抽象语法树（AST）匹配** 对工具调用结果进行严格校验。
* **τ²-bench（Tau²）**：面向真实业务场景的评测集，涵盖 airline、retail、telecom 等领域，重点评估模型在工具辅助下的结构化决策与推理能力。
* **MBPP（Mostly Basic Python Problems）**：经典代码生成基准，用于评估模型的基础编程能力与推理能力。

> 在结果中，凡复现结果与官方技术报告差异超过 **10%** 的点位，均已进行标注（如下划线）。

In [ ]:
from IPython.display import HTML

html = r"""
<div style="max-width: 100%; overflow-x: auto; border: 1px solid #d9d9d9; border-radius: 8px;">
<style>
  table.qwen-bench {
    border-collapse: collapse;
    font-family: Arial, Helvetica, sans-serif;
    font-size: 13px;
    min-width: 1800px;
    width: max-content;
    color: #333;
  }
  ...
</table>
</div>
"""
HTML(html)

**复现偏差分析**

结果观察到，复现结果与官方报告之间的主要差异集中在
- **Multi-Turn 场景**
  * 多步推理带来的误差累积较大，容易导致性能下降；
* **Web Search 场景**
  * 受限于离线 oracle 数据源，信息覆盖不完整；
  * 与真实搜索 API 存在分布差异。

> Qwen2.5 官方技术报告及 leaderboard 中 **未提供 BFCL-v4 的评测结果**。

**评测分数分析**

在具体分数上，复现结果与官方报告整体趋势基本一致，但不同任务类型的偏差程度存在明显差异。相对而言，**Non-Live AST、Live AST 以及部分基础工具调用场景**的复现结果与官方值较为接近，说明在静态或低状态依赖的工具调用设置下，评测流程具有较好的稳定性；而偏差最显著的部分主要集中在 **Multi-Turn** 和 **Web Search** 两类任务上，其中前者往往表现为复现分数相较官方结果出现更大幅度波动，反映出多轮状态管理、上下文累积和中间步骤误差传递对最终成功率影响较强；后者则整体受制于离线 oracle cache 与真实搜索 API 之间的信息覆盖差异，因此更容易出现系统性分数偏移。

总体来看，这些差异更多体现的是**评测环境、工具供给方式与任务执行链路**带来的复现偏差，而不完全等同于模型核心能力本身的变化。
